# Notebook 12 — Cross-Encoder Integration Smoke Test

Run from local Jupyter on the machine hosting FastAPI at `127.0.0.1:8000`, **after** starting the API.
This is a **single-cell** API smoke test. It does not retrain, download or re-evaluate frozen Notebook 09 data, and does not change benchmark files.

**Warning:** The real `rerank` request downloads and loads the Cross-Encoder on first use and may take several minutes on CPU.


In [ ]:
import json
import time
from urllib.request import Request, urlopen

URL = "http://127.0.0.1:8000/search"
QUERY = "πλημμύρες και αποζημιώσεις αγροτών"

def post_search(mode=None):
    body = {"query": QUERY, "k": 5}
    if mode is not None:
        body["mode"] = mode
    request = Request(URL, data=json.dumps(body, ensure_ascii=False).encode("utf-8"),
                      headers={"Content-Type": "application/json"}, method="POST")
    started = time.perf_counter()
    with urlopen(request, timeout=900) as response:
        assert response.status == 200, response.status
        payload = json.loads(response.read().decode("utf-8"))
    elapsed = time.perf_counter() - started
    expected = "dense" if mode is None else mode
    assert payload["mode"] == expected, payload
    hits = payload["results"]
    assert 1 <= len(hits) <= 5, payload
    assert len({h["id"] for h in hits}) == len(hits)
    print(f"{expected}: HTTP 200, {len(hits)} hits, {elapsed:.1f}s; top doc: {hits[0]['id']}")
    return payload

dense = post_search()
hybrid = post_search("hybrid")
rerank = post_search("rerank")
print("PHASE 12 LIVE API SMOKE TEST PASSED (not a relevance or latency benchmark).")
